# Mapping multiple Olist source codes to Texas ZIPs

This notebook maps each Olist `source_code` block to one Texas ZIP using cumulative weighted capacity ranges. The final step joins the mapping back to the customer-level Olist table and writes a new CSV without editing the original source files.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.max_columns = 80

PROJECT_ROOT = Path.cwd()
OLIST_CUSTOMERS_CSV = PROJECT_ROOT / "Olist E-Commerce Dataset" / "olist_customers_dataset.csv"
OLIST_CUSTOMERS_WITH_SOURCE_CSV = PROJECT_ROOT / "Olist E-Commerce Dataset" / "olist_customers_with_source_code.csv"
SOURCE_CODE_SUMMARY_CSV = PROJECT_ROOT / "Synthetic_Customers_and_Orders" / "source_code_summary.csv"
TEXAS_TARGET_WEIGHTS_CSV = PROJECT_ROOT / "Synthetic_Customers_and_Orders" / "texas_target_weights.csv"

OUTPUT_MAPPING_CSV = PROJECT_ROOT / "Establish_source_codes_and_target_weights" / "source_code_to_texas_zip_mapping.csv"
OUTPUT_CUSTOMERS_WITH_ZIP_CSV = PROJECT_ROOT / "Olist E-Commerce Dataset" / "olist_customers_with_assigned_texas_zip.csv"


In [ ]:
# Main three requested dataframes.
olist_customers_df = pd.read_csv(
    OLIST_CUSTOMERS_CSV,
    dtype={"customer_zip_code_prefix": "string"},
)
source_code_summary_df = pd.read_csv(
    SOURCE_CODE_SUMMARY_CSV,
    dtype={"source_code": "string"},
)
texas_weight_df = pd.read_csv(
    TEXAS_TARGET_WEIGHTS_CSV,
    dtype={"zip": "string", "county_geoid": "string"},
)

olist_customers_df["customer_zip_code_prefix"] = olist_customers_df["customer_zip_code_prefix"].str.zfill(5)
texas_weight_df["zip"] = texas_weight_df["zip"].str.zfill(5)

# The raw Olist customer CSV has no source_code column. Use the enriched customer file for the final join.
if "source_code" in olist_customers_df.columns:
    olist_customers_for_mapping_df = olist_customers_df.copy()
elif OLIST_CUSTOMERS_WITH_SOURCE_CSV.exists():
    olist_customers_for_mapping_df = pd.read_csv(
        OLIST_CUSTOMERS_WITH_SOURCE_CSV,
        dtype={
            "customer_zip_code_prefix": "string",
            "zip_group_4": "string",
            "source_code": "string",
            "zip_group_3": "string",
            "zip_group_2": "string",
        },
    )
else:
    raise FileNotFoundError(
        "The raw customer CSV does not contain source_code, and "
        f"{OLIST_CUSTOMERS_WITH_SOURCE_CSV.name} was not found."
    )

print(f"olist_customers_df: {olist_customers_df.shape}")
print(f"source_code_summary_df: {source_code_summary_df.shape}")
print(f"texas_weight_df: {texas_weight_df.shape}")
print(f"olist_customers_for_mapping_df: {olist_customers_for_mapping_df.shape}")


In [ ]:
required_source_cols = {"source_code"}
required_texas_cols = {"zip", "target_zip_capacity"}

missing_source_cols = required_source_cols - set(source_code_summary_df.columns)
missing_texas_cols = required_texas_cols - set(texas_weight_df.columns)

if missing_source_cols:
    raise ValueError(f"source_code_summary_df is missing columns: {sorted(missing_source_cols)}")
if missing_texas_cols:
    raise ValueError(f"texas_weight_df is missing columns: {sorted(missing_texas_cols)}")

if "source_code_count" not in source_code_summary_df.columns:
    if "count" not in source_code_summary_df.columns:
        raise ValueError("source_code_summary_df must contain either 'source_code_count' or 'count'.")
    source_code_summary_df["source_code_count"] = source_code_summary_df["count"]

source_code_summary_df["source_code_count"] = pd.to_numeric(
    source_code_summary_df["source_code_count"],
    errors="raise",
)
texas_weight_df["target_zip_capacity"] = pd.to_numeric(
    texas_weight_df["target_zip_capacity"],
    errors="raise",
)

# Largest source blocks and largest Texas ZIP capacities are packed first.
source_code_summary_df = source_code_summary_df.sort_values(
    ["source_code_count", "source_code"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)
texas_weight_df = texas_weight_df.sort_values(
    ["target_zip_capacity", "zip"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)

source_code_summary_df["source_start"] = source_code_summary_df["source_code_count"].cumsum().shift(fill_value=0)
source_code_summary_df["source_end"] = source_code_summary_df["source_code_count"].cumsum()
source_code_summary_df["source_midpoint"] = (
    source_code_summary_df["source_start"] + source_code_summary_df["source_end"]
) / 2

texas_weight_df["target_start"] = texas_weight_df["target_zip_capacity"].cumsum().shift(fill_value=0)
texas_weight_df["target_end"] = texas_weight_df["target_zip_capacity"].cumsum()

source_total = source_code_summary_df["source_code_count"].sum()
target_total = texas_weight_df["target_zip_capacity"].sum()

print(f"Total Olist source customers: {source_total:,.0f}")
print(f"Total Texas target capacity: {target_total:,.2f}")
print(f"Difference: {target_total - source_total:,.6f}")

display(source_code_summary_df.head())
display(texas_weight_df.head())


In [ ]:
target_ends = texas_weight_df["target_end"].to_numpy()
source_midpoints = source_code_summary_df["source_midpoint"].to_numpy()

# Find the first Texas ZIP cumulative endpoint that contains each source_code midpoint.
assigned_target_positions = np.searchsorted(target_ends, source_midpoints, side="left")
assigned_target_positions = np.clip(assigned_target_positions, 0, len(texas_weight_df) - 1)
assigned_texas_rows = texas_weight_df.iloc[assigned_target_positions].reset_index(drop=True)

mapping_table_df = pd.DataFrame(
    {
        "source_code": source_code_summary_df["source_code"].to_numpy(),
        "source_customer_count": source_code_summary_df["source_code_count"].to_numpy(),
        "source_start": source_code_summary_df["source_start"].to_numpy(),
        "source_end": source_code_summary_df["source_end"].to_numpy(),
        "source_midpoint": source_code_summary_df["source_midpoint"].to_numpy(),
        "assigned_texas_zip": assigned_texas_rows["zip"].to_numpy(),
        "assigned_texas_city": assigned_texas_rows.get("city", pd.Series([pd.NA] * len(assigned_texas_rows))).to_numpy(),
        "assigned_texas_county": assigned_texas_rows.get("county_name", pd.Series([pd.NA] * len(assigned_texas_rows))).to_numpy(),
        "target_zip_capacity": assigned_texas_rows["target_zip_capacity"].to_numpy(),
        "target_start": assigned_texas_rows["target_start"].to_numpy(),
        "target_end": assigned_texas_rows["target_end"].to_numpy(),
    }
)

zip_assignment_summary_df = (
    mapping_table_df.groupby("assigned_texas_zip", as_index=False)
    .agg(
        assigned_source_code_count=("source_code", "count"),
        assigned_customer_count=("source_customer_count", "sum"),
    )
    .merge(
        texas_weight_df[["zip", "target_zip_capacity", "target_start", "target_end"]],
        left_on="assigned_texas_zip",
        right_on="zip",
        how="right",
    )
    .drop(columns="zip")
)
zip_assignment_summary_df["assigned_source_code_count"] = zip_assignment_summary_df["assigned_source_code_count"].fillna(0).astype(int)
zip_assignment_summary_df["assigned_customer_count"] = zip_assignment_summary_df["assigned_customer_count"].fillna(0).astype(int)
zip_assignment_summary_df["capacity_minus_assigned_customers"] = (
    zip_assignment_summary_df["target_zip_capacity"] - zip_assignment_summary_df["assigned_customer_count"]
)

# Sub-1 capacity ZIPs can only receive customers when a whole source_code midpoint lands in their range.
tiny_zip_assignment_df = zip_assignment_summary_df[zip_assignment_summary_df["target_zip_capacity"] < 1].copy()

print(f"Mapped source codes: {len(mapping_table_df):,}")
print(f"Texas ZIPs receiving at least one source_code: {(zip_assignment_summary_df['assigned_source_code_count'] > 0).sum():,}")
print(f"Sub-1 capacity ZIPs receiving at least one source_code: {(tiny_zip_assignment_df['assigned_source_code_count'] > 0).sum():,}")

display(mapping_table_df.head(10))
display(zip_assignment_summary_df.head(10))


In [ ]:
if "source_code" not in olist_customers_for_mapping_df.columns:
    raise ValueError("olist_customers_for_mapping_df must contain source_code before joining the mapping table.")

missing_mapping_source_codes = sorted(
    set(olist_customers_for_mapping_df["source_code"].dropna().unique())
    - set(mapping_table_df["source_code"].dropna().unique())
)
if missing_mapping_source_codes:
    raise ValueError(
        f"{len(missing_mapping_source_codes):,} customer source_codes were not found in mapping_table_df. "
        f"Examples: {missing_mapping_source_codes[:10]}"
    )

olist_customers_with_assigned_zip_df = olist_customers_for_mapping_df.merge(
    mapping_table_df[["source_code", "assigned_texas_zip"]],
    on="source_code",
    how="left",
    validate="many_to_one",
)

missing_assigned_zip_count = olist_customers_with_assigned_zip_df["assigned_texas_zip"].isna().sum()
if missing_assigned_zip_count:
    raise ValueError(f"{missing_assigned_zip_count:,} customer rows did not receive an assigned_texas_zip.")

mapping_table_df.to_csv(OUTPUT_MAPPING_CSV, index=False)
olist_customers_with_assigned_zip_df.to_csv(OUTPUT_CUSTOMERS_WITH_ZIP_CSV, index=False)

print(f"Wrote mapping table: {OUTPUT_MAPPING_CSV}")
print(f"Wrote customer table with assigned Texas ZIPs: {OUTPUT_CUSTOMERS_WITH_ZIP_CSV}")
print(f"Customer rows with assigned_texas_zip: {len(olist_customers_with_assigned_zip_df):,}")

display(olist_customers_with_assigned_zip_df.head())


In [ ]:
quality_check_df = pd.DataFrame(
    {
        "check": [
            "source rows mapped",
            "source customer total",
            "customer rows assigned",
            "unique Texas ZIPs assigned",
            "Texas ZIP target total",
            "sub-1 capacity ZIPs",
            "sub-1 capacity ZIPs assigned",
        ],
        "value": [
            len(mapping_table_df),
            mapping_table_df["source_customer_count"].sum(),
            len(olist_customers_with_assigned_zip_df),
            mapping_table_df["assigned_texas_zip"].nunique(),
            texas_weight_df["target_zip_capacity"].sum(),
            len(tiny_zip_assignment_df),
            (tiny_zip_assignment_df["assigned_source_code_count"] > 0).sum(),
        ],
    }
)

display(quality_check_df)
display(tiny_zip_assignment_df[tiny_zip_assignment_df["assigned_source_code_count"] > 0].head(20))
